# Planning Pattern - Single Agent Framework using LangGraph + Groq

# This notebook demonstrates a planning + replanning agent using:
# - LangGraph (state machine orchestration)
# - Groq - Use LLM of your choice

In [1]:
# --------------------------------------------------
# Setup
# --------------------------------------------------

In [2]:
!pip install langgraph langchain langchain-ollama


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Initialize Groq

In [3]:
import os
from pathlib import Path
import sys

# add parent folder to sys.path so we can import get_api from parent directory
sys.path.insert(0, str(Path.cwd().parent))
import get_api

api_keys = get_api.get_api_keys()
os.environ["GROQ_API_KEY"] = api_keys["GROQ_API_KEY"]
print("✅ Groq API key set!")

✅ Groq API key set!


In [6]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq

llm = ChatGroq(model="mixtral-8x7b-32768", temperature=0)

# --------------------------------------------------
# State Definition
# --------------------------------------------------

class AgentState(TypedDict, total=False):
    input: str
    tasks: List[str]
    completed: List[str]
    credit_score: int
    products: List[str]
    documents: str
    status: str
    plan_history: List[List[str]]

d:\Documents\Agentic_AI_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# --------------------------------------------------
# Tools
# --------------------------------------------------

import random
import re
import ast

def fetch_credit_score(state: AgentState):
    score = random.randint(600, 800)
    print(f"[TOOL] Credit Score: {score}")
    return {"credit_score": score}


def fetch_prime_products(state: AgentState):
    print("[TOOL] Fetching PRIME products")
    return {"products": ["Prime Loan A", "Prime Loan B"]}


def fetch_subprime_products(state: AgentState):
    print("[TOOL] Fetching SUBPRIME products")
    return {"products": ["Subprime Loan X", "Subprime Loan Y"]}


def gather_documents(state: AgentState):
    print("[TOOL] Gathering documents")
    return {"documents": "Collected"}


def submit_application(state: AgentState):
    print("[TOOL] Submitting application")
    return {"status": "Submitted"}

# Optional dynamic branch demo

def request_guarantor(state: AgentState):
    print("[TOOL] Requesting guarantor due to low credit score")
    return {"documents": "Guarantor Added"}

TOOLS = {
    "fetch_credit_score": fetch_credit_score,
    "fetch_prime_products": fetch_prime_products,
    "fetch_subprime_products": fetch_subprime_products,
    "gather_documents": gather_documents,
    "submit_application": submit_application,
    "request_guarantor": request_guarantor
}


Safe Parser

In [8]:
def extract_tasks(response: str):
    try:
        match = re.search(r"\[.*?\]", response, re.DOTALL)
        if match:
            return ast.literal_eval(match.group())
    except Exception as e:
        print("[PARSER ERROR]", e)
    return []

In [9]:
# --------------------------------------------------
# Planner Node (LLM-driven)
# --------------------------------------------------
def planner_node(state: AgentState):
    print("\n" + "="*60)
    print("[PLANNER] Start plan...")

    completed = state.get("completed", [])
    plan_history = state.get("plan_history", [])

    tasks = []
    reason = ""

# ---------------- Deterministic Control ----------------
    if "credit_score" not in state:
        tasks = ["fetch_credit_score"]
        reason = "Credit score missing → must fetch it first"
        stage = "INITIAL PLAN"
    elif "products" not in state:
        if state["credit_score"] < 650:
            tasks = ["request_guarantor"]
            reason = f"Low credit score ({state['credit_score']}) → require guarantor"
        elif state["credit_score"] >= 700:
            tasks = ["fetch_prime_products"]
            reason = f"Credit score {state['credit_score']} ≥ 700 → choose PRIME products"
        else:
            tasks = ["fetch_subprime_products"]
            reason = f"Credit score {state['credit_score']} < 700 → choose SUBPRIME products"
        stage = "REPLAN"

    elif "documents" not in state:
        tasks = ["gather_documents"]
        reason = "Products selected → now gather documents"
        stage = "REPLAN"

    elif "status" not in state:
        tasks = ["submit_application"]
        reason = "All prerequisites met → submit application"
        stage = "REPLAN"

    else:
        tasks = []
        reason = "Goal already achieved"
        stage = "DONE"

    # ---------------- Logging ----------------
    if stage == "INITIAL PLAN":
        print(f"[INITIAL PLAN] {tasks}")
    elif stage == "REPLAN":
        print(f"[REPLAN] {tasks}")

    print(f"[REASON] {reason}")

    # Optional LLM reasoning (adds explainability, not control)
    try:
        explanation_prompt = f"""
Explain in ONE short sentence why this task is chosen.
Task: {tasks}
State: {state}
"""
        llm_reason = llm.invoke(explanation_prompt).content
        print(f"[LLM REASONING] {llm_reason}")
    except Exception:
        pass

    # Track plan history
    plan_history.append(tasks)

    print(f"[PLAN HISTORY] {plan_history}")

    return {
        "tasks": tasks,
        "plan_history": plan_history
    }


In [10]:
# --------------------------------------------------
# Executor Node
# --------------------------------------------------

def executor_node(state: AgentState):
    tasks = state.get("tasks", [])

    if not tasks:
        print("[EXECUTOR] No tasks → forcing replan")
        return state

    task = tasks[0]
    print(f"[EXECUTOR] Running: {task}")

    result = TOOLS[task](state)

    completed = state.get("completed", []) + [task]

    new_state = {
        **state,
        **result,
        "completed": completed,
        "tasks": []
    }

    print(f"[STATE UPDATE] {new_state}")

    return new_state


In [11]:
# --------------------------------------------------
# Decision Node (Continue or End)
# --------------------------------------------------

def should_continue(state: AgentState):
    if state.get("status") == "Submitted":
        print("\n[AGENT] ✅ Goal Achieved")
        return END
    return "planner"

In [12]:
# --------------------------------------------------
# Build Graph
# --------------------------------------------------
graph = StateGraph(AgentState)

graph.add_node("planner", planner_node)
graph.add_node("executor", executor_node)

graph.set_entry_point("planner")

graph.add_edge("planner", "executor")
graph.add_conditional_edges("executor", should_continue)

app = graph.compile()

In [13]:
# --------------------------------------------------
# Run Agent
# --------------------------------------------------
initial_state = {
    "input": "Apply for home loan",
    "tasks": [],
    "completed": [],
    "plan_history": []
}

MAX_STEPS = 10
state = initial_state

for step in range(1, MAX_STEPS + 1):
    state = app.invoke(state)

    if state.get("status") == "Submitted":
        break
else:
    print("\n[AGENT] ❌ Stopped due to max steps (loop protection)")

print("\nFINAL STATE:")
print(state)


[PLANNER] Start plan...
[INITIAL PLAN] ['fetch_credit_score']
[REASON] Credit score missing → must fetch it first
[PLAN HISTORY] [['fetch_credit_score']]
[EXECUTOR] Running: fetch_credit_score
[TOOL] Credit Score: 763
[STATE UPDATE] {'input': 'Apply for home loan', 'tasks': [], 'completed': ['fetch_credit_score'], 'plan_history': [['fetch_credit_score']], 'credit_score': 763}

[PLANNER] Start plan...
[REPLAN] ['fetch_prime_products']
[REASON] Credit score 763 ≥ 700 → choose PRIME products
[PLAN HISTORY] [['fetch_credit_score'], ['fetch_prime_products']]
[EXECUTOR] Running: fetch_prime_products
[TOOL] Fetching PRIME products
[STATE UPDATE] {'input': 'Apply for home loan', 'tasks': [], 'completed': ['fetch_credit_score', 'fetch_prime_products'], 'credit_score': 763, 'plan_history': [['fetch_credit_score'], ['fetch_prime_products']], 'products': ['Prime Loan A', 'Prime Loan B']}

[PLANNER] Start plan...
[REPLAN] ['gather_documents']
[REASON] Products selected → now gather documents
[PLAN